In [1]:
pip install -q google-adk pandas matplotlib

In [2]:
import os

os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6KCK3PWP-lESNfdbSICbXQW6E_U8KPAzOuc8ToIspXrXQ"

In [4]:
from google import genai
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Say hello!"
)

print(response.text)

Hello! How can I help you today?


In [5]:
import os
import pandas as pd
import matplotlib.pyplot as plt

from google.adk.agents import Agent
from google.adk.runners import Runner

In [6]:
sales = pd.DataFrame(
    {
        "Month": [
            "Jan",
            "Feb",
            "Mar",
            "Apr",
            "May",
            "Jun"
        ],
        "Sales": [
            120,
            150,
            170,
            165,
            210,
            240
        ],
        "Profit": [
            30,
            38,
            44,
            42,
            60,
            71
        ]
    }
)

sales.to_csv("sales.csv", index=False)

In [7]:
def load_sales_data() -> str:
    """
    Loads the sales dataset.
    """
    global df
    df = pd.read_csv("sales.csv")

    return f"Loaded {len(df)} rows."

In [8]:
def summarize_dataset() -> str:
    """
    Returns summary statistics.
    """
    return df.describe().to_markdown()

In [9]:
def highest_sales() -> str:
    """
    Finds the month with the highest sales.
    """

    row = df.loc[df["Sales"].idxmax()]

    return (
        f"{row['Month']} "
        f"had the highest sales "
        f"with {row['Sales']} units."
    )

In [10]:
def average_profit() -> str:
    """
    Calculates average profit.
    """

    return str(df["Profit"].mean())

In [11]:
def plot_sales() -> str:
    """
    Creates a line plot of sales.
    """

    plt.figure(figsize=(8,4))

    plt.plot(df["Month"], df["Sales"], marker="o")

    plt.title("Monthly Sales")

    plt.xlabel("Month")

    plt.ylabel("Sales")

    plt.grid(True)

    plt.savefig("sales.png")

    plt.close()

    return "sales.png"

In [12]:
analyst = Agent(
    name="sales_analyst",

    model="gemini-3.6-flash",

    instruction="""
You are a senior data analyst.

Whenever the user asks about sales,
ALWAYS use the available tools.

Never invent statistics.

Explain your reasoning clearly.
""",

    tools=[
        load_sales_data,
        summarize_dataset,
        highest_sales,
        average_profit,
        plot_sales
    ]
)

In [17]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

session_service = InMemorySessionService()

runner = Runner(
    app_name="sales_analyst_app",
    agent=analyst,
    session_service=session_service,
)

In [18]:
APP_NAME = "sales_analyst_app"
USER_ID = "soham"
SESSION_ID = "session_001"

session = session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
)

print(session)

<coroutine object InMemorySessionService.create_session at 0x7af771e61840>


In [19]:
import inspect

print(dir(runner))

['Self', '__aenter__', '__aexit__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_agent_origin_app_name', '_agent_origin_dir', '_app_name_alignment_hint', '_append_new_message_to_session', '_append_user_event', '_cleanup_root_task', '_cleanup_toolsets', '_collect_toolset', '_compute_artifact_delta_for_rewind', '_compute_state_delta_for_rewind', '_consume_event_queue', '_create_invocation_context', '_enforce_app_name_alignment', '_exec_with_plugin', '_extract_resume_inputs', '_find_agent_to_run', '_find_original_user_content', '_find_user_message_for_invocation', '_format_session_not_found_message', '_get_or_create_session', '_get_output_event', '_handle_new